# Volume Fitting with gsplat Gaussians

Fit a 3D volume directly using gsplat's Gaussian primitives (`quat_scale_to_covar_preci`).
No views, no projections — just 3D Gaussians evaluated at voxel positions with MSE loss.

**Approach:**
- Parameterize N Gaussians with means, quaternions, log-scales, intensity logits
- Use `gsplat.quat_scale_to_covar_preci` (CUDA-accelerated) to get precision matrices Σ⁻¹
- Evaluate $f(\mathbf{x}) = \sum_i \alpha_i \exp\left(-\tfrac{1}{2}(\mathbf{x}-\mu_i)^T \Sigma_i^{-1} (\mathbf{x}-\mu_i)\right)$
- Windowed SGD: sample random sub-blocks, cull local Gaussians, MSE loss

## 1. Imports and Setup

In [ ]:
# Add project root to path
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Added {project_root} to Python path")

In [ ]:
import sys, math, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# ── gsplat with pure-PyTorch fallback ────────────────────────────────────
# gsplat needs a CUDA toolkit (nvcc) to JIT-compile its kernels.
# If unavailable, we implement the identical math in pure PyTorch.

_USE_GSPLAT_CUDA = False
try:
    from gsplat import quat_scale_to_covar_preci as _gsplat_quat_scale
    # Test if CUDA backend actually works
    _tq = torch.tensor([[1.,0.,0.,0.]], device='cuda')
    _ts = torch.tensor([[1.,1.,1.]], device='cuda')
    _gsplat_quat_scale(_tq, _ts, compute_covar=True, compute_preci=True)
    _USE_GSPLAT_CUDA = True
    del _tq, _ts
except Exception:
    pass

def _quat_to_rotation(quats: torch.Tensor) -> torch.Tensor:
    """Quaternion (w,x,y,z) → rotation matrix [N, 3, 3]. Pure PyTorch."""
    q = F.normalize(quats, dim=-1)
    w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    R = torch.stack([
        1 - 2*(y*y + z*z),  2*(x*y - w*z),      2*(x*z + w*y),
        2*(x*y + w*z),      1 - 2*(x*x + z*z),  2*(y*z - w*x),
        2*(x*z - w*y),      2*(y*z + w*x),      1 - 2*(x*x + y*y),
    ], dim=-1).reshape(-1, 3, 3)
    return R

def quat_scale_to_preci(quats: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    """(quats [N,4], scales [N,3]) → precision matrix Σ⁻¹ [N,3,3].
    
    Σ  = R @ diag(s²) @ R^T
    Σ⁻¹ = R @ diag(1/s²) @ R^T
    """
    if _USE_GSPLAT_CUDA:
        _, prec = _gsplat_quat_scale(
            quats, scales,
            compute_covar=False, compute_preci=True, triu=False
        )
        return prec
    
    R = _quat_to_rotation(quats)           # [N, 3, 3]
    inv_s2 = 1.0 / (scales ** 2 + 1e-12)  # [N, 3]
    Diag = torch.zeros(quats.shape[0], 3, 3, device=quats.device)
    Diag[:, 0, 0] = inv_s2[:, 0]
    Diag[:, 1, 1] = inv_s2[:, 1]
    Diag[:, 2, 2] = inv_s2[:, 2]
    prec = R @ Diag @ R.transpose(1, 2)
    return prec

# Data loading — import only ProjectionSliceDataset (avoids torchvision issue)
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from inct.dataset_slices import ProjectionSliceDataset

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['image.cmap'] = 'gray'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}  |  PyTorch {torch.__version__}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
backend = "gsplat CUDA" if _USE_GSPLAT_CUDA else "pure PyTorch (gsplat math)"
print(f"Precision backend: {backend}")
print("✅ Imports ready")

## 2. Load Target Volume

In [ ]:
# ── Load same data as GS3D_Demo ─────────────────────────────────────────
DATA_PATH = Path('/myhome/data/sdate/shared/compression_paper/file_1_extracted')
NUM_PROJECTIONS = 50
TARGET_SIZE = 512

dataset = ProjectionSliceDataset(
    folder_path=DATA_PATH,
    num_projections=NUM_PROJECTIONS,
    target_size=TARGET_SIZE,
    normalize_values=True,
    verbose=True,
    cache_volume=True,
    use_attenuation=True,
)

volume_raw = dataset.get_full_volume()
target_volume = volume_raw.permute(2, 0, 1).contiguous()  # (D, H, W)

# Normalise to [0, 1]
vol_min, vol_max = float(target_volume.min()), float(target_volume.max())
vol_range = vol_max - vol_min
target_norm = (target_volume - vol_min) / vol_range

D, H, W = target_norm.shape
print(f"\nTarget volume: ({D}, {H}, {W})")
print(f"Range: [{target_norm.min():.4f}, {target_norm.max():.4f}]")
print(f"Total voxels: {target_norm.numel():,}")
print(f"Original size: {target_norm.numel() * 4 / 1e6:.1f} MB")

## 3. Initialize Gaussian Parameters

Using gsplat's parameterization: means [N,3], quats [N,4], scales [N,3], intensity_logit [N].
- `gsplat.quat_scale_to_covar_preci` converts (quats, scales) → precision matrices on GPU
- Covariance: $\Sigma = R \cdot S \cdot S^T \cdot R^T$, Precision: $\Sigma^{-1} = R \cdot S^{-1} \cdot S^{-1,T} \cdot R^T$

In [ ]:
# ── Gaussian parameters ──────────────────────────────────────────────────
N_INIT = 200_000

torch.manual_seed(42)

# Means: uniform in [0,1]^3 (normalised volume coordinates)
means = torch.rand(N_INIT, 3, device=device)

# Quaternions: identity rotation + small noise
quats = torch.zeros(N_INIT, 4, device=device)
quats[:, 0] = 1.0  # w=1 → identity
quats += 0.01 * torch.randn_like(quats)

# Log-scales: gsplat uses raw scales (not log), but we'll store log-scales
# and exponentiate for quat_scale_to_covar_preci
log_scales = torch.full((N_INIT, 3), -4.0, device=device)  # scale ≈ 0.018

# Intensity logit: sigmoid(logit) → intensity ∈ (0,1)
intensity_logit = torch.full((N_INIT,), 0.5, device=device)

# Make all parameters require gradients
means.requires_grad_(True)
quats.requires_grad_(True)
log_scales.requires_grad_(True)
intensity_logit.requires_grad_(True)

params = {
    'means': means,
    'quats': quats,
    'log_scales': log_scales,
    'intensity_logit': intensity_logit,
}

n_params = sum(p.numel() for p in params.values())
print(f"Gaussians: {N_INIT:,}")
print(f"Parameters: {n_params:,} ({n_params * 4 / 1e6:.2f} MB)")
print(f"  means:          {means.shape}")
print(f"  quats:          {quats.shape}")
print(f"  log_scales:     {log_scales.shape}")
print(f"  intensity_logit: {intensity_logit.shape}")

## 4. Core Evaluation Functions

Direct volumetric evaluation using gsplat's `quat_scale_to_covar_preci` for the precision matrix computation.

In [ ]:
@torch.no_grad()
def evaluate_gaussians_at_points(
    coords: torch.Tensor,     # (B, 3) — normalised [0,1]^3 coordinates
    means: torch.Tensor,      # (N, 3)
    quats: torch.Tensor,      # (N, 4)
    log_scales: torch.Tensor, # (N, 3)
    intensity_logit: torch.Tensor, # (N,)
    cutoff_sigma: float = 3.0,
    tile_size: int = 1024,
) -> torch.Tensor:
    """Evaluate Gaussian mixture at given 3D coordinates (inference only, no grad).
    
    Uses quat_scale_to_preci + bounding-box culling + tiled evaluation.
    Returns: (B,) tensor of reconstructed values at each coordinate.
    """
    B = coords.shape[0]
    scales = torch.exp(log_scales)
    intensities = torch.sigmoid(intensity_logit)
    prec = quat_scale_to_preci(quats, scales)  # [N, 3, 3]
    
    # Bounding box culling
    coord_min = coords.min(dim=0).values
    coord_max = coords.max(dim=0).values
    margin = cutoff_sigma * scales
    mask = ((means - margin) < coord_max).all(dim=1) & ((means + margin) > coord_min).all(dim=1)
    idx_local = mask.nonzero(as_tuple=True)[0]
    
    if idx_local.shape[0] == 0:
        return torch.zeros(B, device=coords.device)
    
    mu_l = means[idx_local]
    prec_l = prec[idx_local]
    int_l = intensities[idx_local]
    
    out = torch.zeros(B, device=coords.device)
    for start in range(0, B, tile_size):
        end = min(start + tile_size, B)
        c = coords[start:end]
        diff = c[:, None, :] - mu_l[None, :, :]
        tmp = torch.einsum('tnj,njk->tnk', diff, prec_l)
        mahal = (tmp * diff).sum(dim=-1)
        vals = torch.exp(-0.5 * mahal) * (mahal < cutoff_sigma**2 * 3).float() * int_l[None, :]
        out[start:end] = vals.sum(dim=1)
    
    return out


def train_step_tiled(
    coords: torch.Tensor,      # (B, 3)
    target: torch.Tensor,       # (B,)
    means: torch.Tensor,
    quats: torch.Tensor,
    log_scales: torch.Tensor,
    intensity_logit: torch.Tensor,
    cutoff_sigma: float = 3.0,
    tile_size: int = 2048,
) -> float:
    """Forward + backward with per-tile gradient accumulation.
    
    Key insight: compute loss per tile and call backward(retain_graph=True)
    so that tile-specific intermediates (diff, mahal, etc.) are freed after
    each tile, while the shared precision graph is retained.
    """
    B = coords.shape[0]
    scales = torch.exp(log_scales)
    intensities = torch.sigmoid(intensity_logit)
    prec = quat_scale_to_preci(quats, scales)  # [N, 3, 3] — shared graph
    
    # Bounding box culling (detached — no autograd overhead)
    with torch.no_grad():
        coord_min = coords.min(dim=0).values
        coord_max = coords.max(dim=0).values
        margin = cutoff_sigma * scales
        mask = ((means - margin) < coord_max).all(dim=1) & ((means + margin) > coord_min).all(dim=1)
        idx_local = mask.nonzero(as_tuple=True)[0]
    
    if idx_local.shape[0] == 0:
        return 0.0
    
    mu_l = means[idx_local]
    prec_l = prec[idx_local]
    int_l = intensities[idx_local]
    
    total_loss = 0.0
    n_tiles = (B + tile_size - 1) // tile_size
    
    for i, start in enumerate(range(0, B, tile_size)):
        end = min(start + tile_size, B)
        T = end - start
        c = coords[start:end]
        t = target[start:end]
        
        diff = c[:, None, :] - mu_l[None, :, :]  # [T, K, 3]
        tmp = torch.einsum('tnj,njk->tnk', diff, prec_l)
        mahal = (tmp * diff).sum(dim=-1)
        mask_cut = mahal < (cutoff_sigma ** 2 * 3)
        vals = (torch.exp(-0.5 * mahal) * mask_cut.float() * int_l[None, :]).sum(dim=1)
        
        tile_loss = F.mse_loss(vals, t) * (T / B)  # weighted by tile fraction
        
        is_last = (i == n_tiles - 1)
        tile_loss.backward(retain_graph=not is_last)
        
        total_loss += tile_loss.item()
    
    return total_loss


def sample_window(volume_shape, window_size):
    """Sample a random window origin. Returns (origin, coords)."""
    D, H, W = volume_shape
    wd, wh, ww = window_size
    
    d0 = torch.randint(0, max(1, D - wd + 1), (1,)).item()
    h0 = torch.randint(0, max(1, H - wh + 1), (1,)).item()
    w0 = torch.randint(0, max(1, W - ww + 1), (1,)).item()
    
    dg = torch.arange(d0, d0 + wd, device=device).float() / max(D - 1, 1)
    hg = torch.arange(h0, h0 + wh, device=device).float() / max(H - 1, 1)
    wg = torch.arange(w0, w0 + ww, device=device).float() / max(W - 1, 1)
    
    grid_d, grid_h, grid_w = torch.meshgrid(dg, hg, wg, indexing='ij')
    coords = torch.stack([grid_d, grid_h, grid_w], dim=-1).reshape(-1, 3)
    
    return (d0, h0, w0), coords


print("✅ Functions defined")
backend = "gsplat CUDA" if _USE_GSPLAT_CUDA else "pure PyTorch"
print(f"  Precision backend: {backend}")
print(f"  train_step_tiled: per-tile gradient accumulation (memory-efficient)")

## 5. Training — Windowed SGD with gsplat Precision Matrices

In [ ]:
# ── Training config ──────────────────────────────────────────────────────
NUM_EPOCHS    = 5000
WINDOW_SIZE   = (32, 32, 32)         # Spatial window per step
VOXELS_PER_STEP = 65_536              # Sub-sample from window (64K voxels)
CUTOFF_SIGMA  = 3.0
TILE_SIZE     = 2048                  # Voxels per tile in per-tile backward
LOG_INTERVAL  = 200
EVAL_INTERVAL = 1000
EVAL_SAMPLES  = 100_000

# Learning rates
LR_MEANS      = 2e-4
LR_QUATS      = 1e-3
LR_SCALES     = 5e-3
LR_INTENSITY  = 5e-2
LR_DECAY      = 0.05  # Final LR = initial * LR_DECAY

# Pruning
PRUNE_FROM    = 500
PRUNE_INTERVAL = 200
PRUNE_THRESH  = 0.01

# Move target to GPU
target_gpu = target_norm.to(device)
target_flat = target_gpu.reshape(-1)  # (D*H*W,)

# ── Optimizer: separate param groups ────────────────────────────────────
optimizer = torch.optim.AdamW([
    {'params': [means],          'lr': LR_MEANS,     'name': 'means'},
    {'params': [quats],          'lr': LR_QUATS,     'name': 'quats'},
    {'params': [log_scales],     'lr': LR_SCALES,    'name': 'scales'},
    {'params': [intensity_logit],'lr': LR_INTENSITY,  'name': 'intensity'},
], weight_decay=0.0)

# Exponential LR schedule
gamma = (LR_DECAY) ** (1.0 / max(NUM_EPOCHS, 1))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

print(f"Training config:")
print(f"  Epochs: {NUM_EPOCHS}, Window: {WINDOW_SIZE}, Voxels/step: {VOXELS_PER_STEP}")
print(f"  Tile size: {TILE_SIZE}, Cutoff: {CUTOFF_SIGMA}σ, LR decay: {LR_DECAY}")
print(f"  Pruning: from epoch {PRUNE_FROM}, interval {PRUNE_INTERVAL}, thresh {PRUNE_THRESH}")

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────
loss_history = []
metrics_history = []
t_start = time.time()

print("🚀 Starting training...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    
    # --- Sample random window ---
    (d0, h0, w0), coords = sample_window(
        (D, H, W), WINDOW_SIZE
    )
    
    # Sub-sample voxels from window
    V = coords.shape[0]
    if V > VOXELS_PER_STEP:
        idx = torch.randperm(V, device=device)[:VOXELS_PER_STEP]
        coords_sub = coords[idx]
    else:
        coords_sub = coords
        idx = torch.arange(V, device=device)
    
    # Get target values for these voxels
    wd, wh, ww = WINDOW_SIZE
    target_window = target_gpu[d0:d0+wd, h0:h0+wh, w0:w0+ww].reshape(-1)
    if V > VOXELS_PER_STEP:
        target_sub = target_window[idx]
    else:
        target_sub = target_window
    
    # --- Forward + backward with per-tile gradient accumulation ---
    loss_val = train_step_tiled(
        coords_sub, target_sub,
        means, quats, log_scales, intensity_logit,
        cutoff_sigma=CUTOFF_SIGMA, tile_size=TILE_SIZE,
    )
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(
        [means, quats, log_scales, intensity_logit], max_norm=1.0
    )
    
    optimizer.step()
    scheduler.step()
    
    loss_history.append(loss_val)
    
    # --- Pruning ---
    n_pruned = 0
    if epoch >= PRUNE_FROM and epoch % PRUNE_INTERVAL == 0:
        with torch.no_grad():
            intensities_now = torch.sigmoid(intensity_logit)
            keep_mask = intensities_now > PRUNE_THRESH
            n_before = means.shape[0]
            
            if keep_mask.sum() < n_before:
                means_new = means.data[keep_mask].clone().requires_grad_(True)
                quats_new = quats.data[keep_mask].clone().requires_grad_(True)
                log_scales_new = log_scales.data[keep_mask].clone().requires_grad_(True)
                intensity_logit_new = intensity_logit.data[keep_mask].clone().requires_grad_(True)
                
                n_pruned = n_before - means_new.shape[0]
                
                means = means_new
                quats = quats_new
                log_scales = log_scales_new
                intensity_logit = intensity_logit_new
                
                optimizer = torch.optim.AdamW([
                    {'params': [means],          'lr': optimizer.param_groups[0]['lr'], 'name': 'means'},
                    {'params': [quats],          'lr': optimizer.param_groups[1]['lr'], 'name': 'quats'},
                    {'params': [log_scales],     'lr': optimizer.param_groups[2]['lr'], 'name': 'scales'},
                    {'params': [intensity_logit],'lr': optimizer.param_groups[3]['lr'], 'name': 'intensity'},
                ], weight_decay=0.0)
                scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    # --- Logging ---
    if epoch % LOG_INTERVAL == 0 or epoch == NUM_EPOCHS - 1:
        N_cur = means.shape[0]
        extra = f"  pruned={n_pruned}" if n_pruned > 0 else ""
        print(f"[Epoch {epoch:5d}/{NUM_EPOCHS}]  loss={loss_val:.6f}  N={N_cur:,}{extra}")
    
    # --- Evaluation ---
    if epoch % EVAL_INTERVAL == 0 and epoch > 0:
        with torch.no_grad():
            eval_idx = torch.randperm(target_flat.shape[0], device=device)[:EVAL_SAMPLES]
            d_idx = eval_idx // (H * W)
            h_idx = (eval_idx % (H * W)) // W
            w_idx = eval_idx % W
            eval_coords = torch.stack([
                d_idx.float() / max(D-1, 1),
                h_idx.float() / max(H-1, 1),
                w_idx.float() / max(W-1, 1),
            ], dim=1)
            
            eval_target = target_flat[eval_idx]
            eval_pred = evaluate_gaussians_at_points(
                eval_coords, means, quats, log_scales, intensity_logit,
                cutoff_sigma=CUTOFF_SIGMA, tile_size=4096,
            )
            
            eval_mse = F.mse_loss(eval_pred, eval_target).item()
            eval_psnr = -10 * math.log10(eval_mse + 1e-10)
            
            model_mb = means.shape[0] * 11 * 4 / 1e6
            metrics_history.append({
                'epoch': epoch, 'psnr': eval_psnr, 'mse': eval_mse,
                'n_gaussians': means.shape[0], 'model_mb': model_mb,
            })
            print(f"         PSNR={eval_psnr:.2f} dB  MSE={eval_mse:.6f}  model={model_mb:.2f} MB")

elapsed = time.time() - t_start
N_final = means.shape[0]

# Final evaluation
with torch.no_grad():
    eval_idx = torch.randperm(target_flat.shape[0], device=device)[:EVAL_SAMPLES]
    d_idx = eval_idx // (H * W)
    h_idx = (eval_idx % (H * W)) // W
    w_idx = eval_idx % W
    eval_coords = torch.stack([
        d_idx.float() / max(D-1, 1),
        h_idx.float() / max(H-1, 1),
        w_idx.float() / max(W-1, 1),
    ], dim=1)
    eval_target = target_flat[eval_idx]
    eval_pred = evaluate_gaussians_at_points(
        eval_coords, means, quats, log_scales, intensity_logit,
        cutoff_sigma=CUTOFF_SIGMA, tile_size=4096,
    )
    final_mse = F.mse_loss(eval_pred, eval_target).item()
    final_psnr = -10 * math.log10(final_mse + 1e-10)

print(f"\n✅ Training complete in {elapsed:.1f}s")
print(f"   Final loss: {loss_history[-1]:.6f}")
print(f"   Final PSNR: {final_psnr:.2f} dB  (sampled)")
print(f"   Final MSE:  {final_mse:.6f}")
print(f"   Gaussians:  {N_final:,}")
print(f"   Model size: {N_final * 11 * 4 / 1e6:.2f} MB")

## 6. Full Volume Reconstruction & Metrics

In [ ]:
# ── Full-volume reconstruction (block-by-block) ─────────────────────────
import numpy as np
from skimage.metrics import structural_similarity as ssim

@torch.no_grad()
def render_full_volume(means, quats, log_scales, intensity_logit,
                       shape, cutoff_sigma=3.0, block_size=512*512):
    """Reconstruct the entire volume block-by-block."""
    D, H, W = shape
    volume = torch.zeros(D * H * W, device=means.device)
    
    # Precompute precision matrices once (using our fallback-aware function)
    scales = torch.exp(log_scales)
    precis = quat_scale_to_preci(quats, scales)  # [N, 3, 3]
    intensities = torch.sigmoid(intensity_logit)
    
    total_voxels = D * H * W
    n_blocks = (total_voxels + block_size - 1) // block_size
    
    for block_idx in range(n_blocks):
        start = block_idx * block_size
        end = min(start + block_size, total_voxels)
        
        # Convert flat indices to normalised coordinates
        flat_idx = torch.arange(start, end, device=means.device)
        d_idx = flat_idx // (H * W)
        h_idx = (flat_idx % (H * W)) // W
        w_idx = flat_idx % W
        
        coords = torch.stack([
            d_idx.float() / max(D-1, 1),
            h_idx.float() / max(H-1, 1),
            w_idx.float() / max(W-1, 1),
        ], dim=1)  # [B, 3]
        
        # Bounding-box culling
        sigma_extent = cutoff_sigma * scales.max(dim=1).values  # [N]
        coords_min = coords.min(dim=0).values  # [3]
        coords_max = coords.max(dim=0).values  # [3]
        
        in_range = (
            (means[:, 0] + sigma_extent >= coords_min[0]) &
            (means[:, 0] - sigma_extent <= coords_max[0]) &
            (means[:, 1] + sigma_extent >= coords_min[1]) &
            (means[:, 1] - sigma_extent <= coords_max[1]) &
            (means[:, 2] + sigma_extent >= coords_min[2]) &
            (means[:, 2] - sigma_extent <= coords_max[2])
        )
        
        if in_range.sum() == 0:
            continue
        
        m_sel = means[in_range]       # [K, 3]
        p_sel = precis[in_range]      # [K, 3, 3]
        i_sel = intensities[in_range] # [K]
        
        # Tiled evaluation
        B = coords.shape[0]
        TILE = 256
        block_vals = torch.zeros(B, device=means.device)
        
        for t in range(0, B, TILE):
            t_end = min(t + TILE, B)
            c_tile = coords[t:t_end]  # [T, 3]
            
            diff = c_tile.unsqueeze(1) - m_sel.unsqueeze(0)  # [T, K, 3]
            mahal = torch.einsum('tkd,kde,tke->tk', diff, p_sel, diff)  # [T, K]
            valid = mahal < (cutoff_sigma ** 2)
            
            gauss = torch.exp(-0.5 * mahal) * valid.float()  # [T, K]
            block_vals[t:t_end] = (gauss * i_sel.unsqueeze(0)).sum(dim=1)
        
        volume[start:end] = block_vals
    
    return volume.reshape(D, H, W)

print("Reconstructing full volume...")
t0 = time.time()
recon = render_full_volume(
    means, quats, log_scales, intensity_logit,
    (D, H, W), cutoff_sigma=CUTOFF_SIGMA, block_size=512*512
)
recon_np = recon.cpu().numpy()
target_np = target_gpu.cpu().numpy()
recon_time = time.time() - t0
print(f"Reconstruction done in {recon_time:.1f}s")

# Clamp to valid range
recon_np = np.clip(recon_np, 0.0, 1.0)

# ── Metrics ──────────────────────────────────────────────────────────────
full_mse = np.mean((recon_np - target_np) ** 2)
full_psnr = -10 * np.log10(full_mse + 1e-10)

# SSIM per slice (depth axis)
ssim_vals = []
for d in range(D):
    s = ssim(target_np[d], recon_np[d], data_range=1.0)
    ssim_vals.append(s)
mean_ssim = np.mean(ssim_vals)

# Compression ratio
original_bytes = D * H * W * 2  # int16
model_params = means.shape[0] * 11  # 3+4+3+1
model_bytes = model_params * 4  # float32
compression_ratio = original_bytes / model_bytes

print(f"\n{'='*50}")
print(f"FULL-VOLUME METRICS (gsplat)")
print(f"{'='*50}")
print(f"  MSE:               {full_mse:.6f}")
print(f"  PSNR:              {full_psnr:.2f} dB")
print(f"  SSIM:              {mean_ssim:.4f}")
print(f"  Gaussians:         {means.shape[0]:,}")
print(f"  Model size:        {model_bytes / 1e6:.2f} MB")
print(f"  Original size:     {original_bytes / 1e6:.2f} MB")
print(f"  Compression ratio: {compression_ratio:.2f}×")

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt

# ── Loss curve ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].semilogy(loss_history)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss (gsplat)')
axes[0].grid(True, alpha=0.3)

# PSNR history
if metrics_history:
    epochs_eval = [m['epoch'] for m in metrics_history]
    psnrs = [m['psnr'] for m in metrics_history]
    axes[1].plot(epochs_eval, psnrs, 'o-')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('PSNR (dB)')
    axes[1].set_title('Evaluation PSNR (sampled)')
    axes[1].grid(True, alpha=0.3)

# Gaussians over time
if metrics_history:
    n_gs = [m['n_gaussians'] for m in metrics_history]
    axes[2].plot(epochs_eval, [n/1000 for n in n_gs], 'o-', color='green')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Gaussians (K)')
    axes[2].set_title('Gaussian Count')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Slice comparisons ────────────────────────────────────────────────────
# Show 4 evenly-spaced depth slices
slice_indices = np.linspace(0, D-1, 4, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, si in enumerate(slice_indices):
    axes[0, i].imshow(target_np[si], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Target (z={si})')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(recon_np[si], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'gsplat Recon (z={si})')
    axes[1, i].axis('off')

plt.suptitle(f'gsplat Volume Fitting — PSNR={full_psnr:.2f} dB, SSIM={mean_ssim:.4f}, CR={compression_ratio:.2f}×', fontsize=14)
plt.tight_layout()
plt.show()

# ── Error map ────────────────────────────────────────────────────────────
mid_slice = D // 2
error = np.abs(recon_np[mid_slice] - target_np[mid_slice])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(target_np[mid_slice], cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Target (z={mid_slice})')
axes[0].axis('off')

axes[1].imshow(recon_np[mid_slice], cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'gsplat Recon (z={mid_slice})')
axes[1].axis('off')

im = axes[2].imshow(error, cmap='hot', vmin=0, vmax=0.1)
axes[2].set_title(f'Absolute Error (z={mid_slice})')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

# ── Per-slice SSIM ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(D), ssim_vals, 'o-')
ax.set_xlabel('Depth Slice')
ax.set_ylabel('SSIM')
ax.set_title('Per-Slice SSIM')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()